In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\DS Project\credit-portfolio-risk-modeling"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import pandas as pd
import gc

from config.paths import (
    ORIG_PARQUET_DIR,
    PERF_PARQUET_DIR,
    AUDIT_DIR,
    DUCKDB_TEMP_DIR
)

# Configure DuckDB
con = duckdb.connect()

# Limit memory usage
con.execute("SET memory_limit = '4GB'")

# Use E drive for temporary processing
temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")

con.execute(
    f"SET temp_directory = '{temp_path}'"
)

print("DuckDB version:", duckdb.__version__)
print("Origination directory:", ORIG_PARQUET_DIR)
print("Performance directory:", PERF_PARQUET_DIR)
print("DuckDB memory limit: 4GB")

DuckDB version: 2.0.0
Origination directory: E:\DS_project_backup\credit-risk-portfolio\processed\origination
Performance directory: E:\DS_project_backup\credit-risk-portfolio\processed\performance
DuckDB memory limit: 4GB


In [2]:
orig_glob = (
    ORIG_PARQUET_DIR / "orig_*.parquet"
).as_posix()

orig_integrity = con.execute(
    """
    SELECT
        COUNT(*) AS total_records,

        COUNT(loan_identifier)
            AS non_missing_loan_ids,

        COUNT(DISTINCT loan_identifier)
            AS unique_loan_ids,

        COUNT(*) FILTER (
            WHERE loan_identifier IS NULL
        ) AS missing_loan_ids

    FROM read_parquet(?)
    """,
    [orig_glob]
).df()

display(orig_integrity)

,total_records,non_missing_loan_ids,unique_loan_ids,missing_loan_ids
0,11128442,11128442,11128442,0


In [3]:
duplicate_check = con.execute(
    """
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT loan_identifier)
            AS unique_loan_ids,

        COUNT(*) - COUNT(DISTINCT loan_identifier)
            AS duplicate_excess_records

    FROM read_parquet(?)
    """,
    [orig_glob]
).df()

display(duplicate_check)

,total_records,unique_loan_ids,duplicate_excess_records
0,11128442,11128442,0


In [6]:
# Close previous connection if it is still open
try:
    con.close()
except Exception:
    pass

import duckdb

con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

temp_path = (
    DUCKDB_TEMP_DIR.as_posix()
    .replace("'", "''")
)

con.execute(
    f"SET temp_directory = '{temp_path}'"
)

print("Memory limit:", con.execute(
    "SELECT current_setting('memory_limit')"
).fetchone()[0])

print("Threads:", con.execute(
    "SELECT current_setting('threads')"
).fetchone()[0])

print("Temporary directory:", DUCKDB_TEMP_DIR)

Memory limit: 2.7 GiB
Threads: 2
Temporary directory: E:\DS_project_backup\credit-risk-portfolio\duckdb_temp


In [7]:
perf_2012q4 = (
    PERF_PARQUET_DIR / "perf_2012Q4.parquet"
).as_posix()

pair_check_2012q4 = con.execute(
    """
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT (
            loan_identifier,
            period
        )) AS unique_loan_month_pairs,

        COUNT(*) - COUNT(DISTINCT (
            loan_identifier,
            period
        )) AS duplicate_excess_pairs

    FROM read_parquet(?)

    WHERE loan_identifier IS NOT NULL
      AND period IS NOT NULL
    """,
    [perf_2012q4]
).df()

display(pair_check_2012q4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,unique_loan_month_pairs,duplicate_excess_pairs
0,56807882,56807882,0


In [8]:
import gc
import pandas as pd
import duckdb

perf_pair_audit = []

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"

        file_path = (
            PERF_PARQUET_DIR /
            f"perf_{vintage}.parquet"
        )

        print(f"\nChecking {vintage}...")

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute(
                "SET preserve_insertion_order = false"
            )

            temp_path = (
                DUCKDB_TEMP_DIR.as_posix()
                .replace("'", "''")
            )

            con.execute(
                f"SET temp_directory = '{temp_path}'"
            )

            result = con.execute(
                """
                SELECT
                    COUNT(*) AS total_records,

                    COUNT(*) FILTER (
                        WHERE loan_identifier IS NOT NULL
                          AND period IS NOT NULL
                    ) AS valid_key_records,

                    COUNT(DISTINCT (
                        loan_identifier,
                        period
                    )) FILTER (
                        WHERE loan_identifier IS NOT NULL
                          AND period IS NOT NULL
                    ) AS unique_loan_month_pairs

                FROM read_parquet(?)
                """,
                [str(file_path)]
            ).fetchone()

            total_records = result[0]
            valid_records = result[1]
            unique_pairs = result[2]

            duplicate_excess = (
                valid_records - unique_pairs
            )

            record = {
                "vintage": vintage,
                "total_records": total_records,
                "valid_key_records": valid_records,
                "unique_pairs": unique_pairs,
                "duplicate_excess": duplicate_excess,
                "status": (
                    "passed"
                    if duplicate_excess == 0
                    else "duplicates_found"
                )
            }

            perf_pair_audit.append(record)

            print(f"Records: {total_records:,}")
            print(f"Unique pairs: {unique_pairs:,}")
            print(f"Duplicate excess: {duplicate_excess:,}")

        finally:
            con.close()
            gc.collect()

        # Save progress after every vintage
        pd.DataFrame(perf_pair_audit).to_csv(
            AUDIT_DIR / "performance_pair_integrity.csv",
            index=False,
            encoding="utf-8-sig"
        )

perf_pair_audit_df = pd.DataFrame(perf_pair_audit)

display(perf_pair_audit_df)

print(
    "Total records:",
    f"{perf_pair_audit_df['total_records'].sum():,}"
)

print(
    "Total duplicate excess:",
    f"{perf_pair_audit_df['duplicate_excess'].sum():,}"
)


Checking 2006Q1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Records: 20,641,773
Unique pairs: 20,641,773
Duplicate excess: 0

Checking 2006Q2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Records: 19,232,256
Unique pairs: 19,232,256
Duplicate excess: 0

Checking 2006Q3...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (49 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Records: 56,807,882
Unique pairs: 56,807,882
Duplicate excess: 0


,vintage,total_records,valid_key_records,unique_pairs,duplicate_excess,status
0,2006Q1,20641773,20641773,20641773,0,passed
1,2006Q2,19232256,19232256,19232256,0,passed
2,2006Q3,17091022,17091022,17091022,0,passed
3,2006Q4,19600060,19600060,19600060,0,passed
4,2007Q1,19455523,19455523,19455523,0,passed
5,2007Q2,22314866,22314866,22314866,0,passed
6,2007Q3,15847264,15847264,15847264,0,passed
7,2007Q4,16954114,16954114,16954114,0,passed
8,2008Q1,23849747,23849747,23849747,0,passed
9,2008Q2,19540165,19540165,19540165,0,passed


Total records: 769,725,294
Total duplicate excess: 0


In [9]:
import duckdb

con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")

con.execute(
    f"SET temp_directory = '{temp_path}'"
)

orig_glob = (
    ORIG_PARQUET_DIR / "orig_*.parquet"
).as_posix()

# Count Loan IDs appearing in multiple Origination files
orig_cross_vintage = con.execute(
    """
    SELECT
        COUNT(*) AS total_unique_loan_ids,

        COUNT(*) FILTER (
            WHERE vintage_count > 1
        ) AS loans_in_multiple_vintages

    FROM (
        SELECT
            loan_identifier,
            COUNT(DISTINCT vintage) AS vintage_count

        FROM (
            SELECT
                loan_identifier,

                regexp_extract(
                    filename,
                    'orig_(\\d{4}Q\\d)\\.parquet',
                    1
                ) AS vintage

            FROM read_parquet(
                ?,
                filename = true
            )
        )

        WHERE loan_identifier IS NOT NULL

        GROUP BY loan_identifier
    )
    """,
    [orig_glob]
).df()

display(orig_cross_vintage)

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_unique_loan_ids,loans_in_multiple_vintages
0,11128442,0


In [10]:
con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
con.execute(f"SET temp_directory = '{temp_path}'")

perf_glob = (
    PERF_PARQUET_DIR / "perf_*.parquet"
).as_posix()

perf_cross_vintage = con.execute(
    """
    SELECT
        COUNT(*) AS total_unique_loan_ids,

        COUNT(*) FILTER (
            WHERE vintage_count > 1
        ) AS loans_in_multiple_vintages

    FROM (
        SELECT
            loan_identifier,
            COUNT(DISTINCT vintage) AS vintage_count

        FROM (
            SELECT
                loan_identifier,

                regexp_extract(
                    filename,
                    'perf_(\\d{4}Q\\d)\\.parquet',
                    1
                ) AS vintage

            FROM read_parquet(
                ?,
                filename = true
            )
        )

        WHERE loan_identifier IS NOT NULL

        GROUP BY loan_identifier
    )
    """,
    [perf_glob]
).df()

display(perf_cross_vintage)

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_unique_loan_ids,loans_in_multiple_vintages
0,11128442,0


In [11]:
import duckdb

orig_file = (
    ORIG_PARQUET_DIR / "orig_2012Q4.parquet"
)

perf_file = (
    PERF_PARQUET_DIR / "perf_2012Q4.parquet"
)

con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
con.execute(f"SET temp_directory = '{temp_path}'")

try:
    result = con.execute(
        """
        WITH
        orig_ids AS (
            SELECT DISTINCT loan_identifier
            FROM read_parquet(?)
            WHERE loan_identifier IS NOT NULL
        ),

        perf_ids AS (
            SELECT DISTINCT loan_identifier
            FROM read_parquet(?)
            WHERE loan_identifier IS NOT NULL
        )

        SELECT
            (SELECT COUNT(*) FROM orig_ids)
                AS origination_unique_loans,

            (SELECT COUNT(*) FROM perf_ids)
                AS performance_unique_loans,

            (
                SELECT COUNT(*)
                FROM perf_ids p
                ANTI JOIN orig_ids o
                USING (loan_identifier)
            ) AS performance_without_origination,

            (
                SELECT COUNT(*)
                FROM orig_ids o
                ANTI JOIN perf_ids p
                USING (loan_identifier)
            ) AS origination_without_performance
        """,
        [str(orig_file), str(perf_file)]
    ).df()

    display(result)

finally:
    con.close()

,origination_unique_loans,performance_unique_loans,performance_without_origination,origination_without_performance
0,592713,592713,0,0


In [12]:
import duckdb
import pandas as pd
import gc

referential_audit = []

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"

        orig_file = (
            ORIG_PARQUET_DIR /
            f"orig_{vintage}.parquet"
        )

        perf_file = (
            PERF_PARQUET_DIR /
            f"perf_{vintage}.parquet"
        )

        print(f"\nChecking {vintage}...")

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute(
                "SET preserve_insertion_order = false"
            )

            temp_path = (
                DUCKDB_TEMP_DIR.as_posix()
                .replace("'", "''")
            )

            con.execute(
                f"SET temp_directory = '{temp_path}'"
            )

            result = con.execute(
                """
                WITH
                orig_ids AS (
                    SELECT DISTINCT loan_identifier
                    FROM read_parquet(?)
                    WHERE loan_identifier IS NOT NULL
                ),

                perf_ids AS (
                    SELECT DISTINCT loan_identifier
                    FROM read_parquet(?)
                    WHERE loan_identifier IS NOT NULL
                )

                SELECT
                    (SELECT COUNT(*) FROM orig_ids)
                        AS origination_loans,

                    (SELECT COUNT(*) FROM perf_ids)
                        AS performance_loans,

                    (
                        SELECT COUNT(*)
                        FROM perf_ids p
                        ANTI JOIN orig_ids o
                        USING (loan_identifier)
                    ) AS performance_without_origination,

                    (
                        SELECT COUNT(*)
                        FROM orig_ids o
                        ANTI JOIN perf_ids p
                        USING (loan_identifier)
                    ) AS origination_without_performance
                """,
                [str(orig_file), str(perf_file)]
            ).fetchone()

            record = {
                "vintage": vintage,
                "origination_loans": result[0],
                "performance_loans": result[1],
                "performance_without_origination": result[2],
                "origination_without_performance": result[3],
                "status": (
                    "passed"
                    if result[2] == 0 and result[3] == 0
                    else "mismatch_found"
                )
            }

            referential_audit.append(record)

            print("Origination:", f"{result[0]:,}")
            print("Performance:", f"{result[1]:,}")
            print("Performance without Origination:", result[2])
            print("Origination without Performance:", result[3])

        finally:
            con.close()
            gc.collect()

        # Save progress after each vintage
        pd.DataFrame(referential_audit).to_csv(
            AUDIT_DIR / "referential_integrity_audit.csv",
            index=False,
            encoding="utf-8-sig"
        )

referential_audit_df = pd.DataFrame(referential_audit)

display(referential_audit_df)

print("\nTotal Origination loans:",
      f"{referential_audit_df['origination_loans'].sum():,}")

print("\nTotal Performance loans:",
      f"{referential_audit_df['performance_loans'].sum():,}")

print("\nTotal orphan Performance loans:",
      f"{referential_audit_df['performance_without_origination'].sum():,}")

print("\nTotal Origination loans without Performance:",
      f"{referential_audit_df['origination_without_performance'].sum():,}")


Checking 2006Q1...
Origination: 298,574
Performance: 298,574
Performance without Origination: 0
Origination without Performance: 0

Checking 2006Q2...
Origination: 308,208
Performance: 308,208
Performance without Origination: 0
Origination without Performance: 0

Checking 2006Q3...
Origination: 283,975
Performance: 283,975
Performance without Origination: 0
Origination without Performance: 0

Checking 2006Q4...
Origination: 302,787
Performance: 302,787
Performance without Origination: 0
Origination without Performance: 0

Checking 2007Q1...
Origination: 302,331
Performance: 302,331
Performance without Origination: 0
Origination without Performance: 0

Checking 2007Q2...
Origination: 354,055
Performance: 354,055
Performance without Origination: 0
Origination without Performance: 0

Checking 2007Q3...
Origination: 284,980
Performance: 284,980
Performance without Origination: 0
Origination without Performance: 0

Checking 2007Q4...
Origination: 296,826
Performance: 296,826
Performance wi

,vintage,origination_loans,performance_loans,performance_without_origination,origination_without_performance,status
0,2006Q1,298574,298574,0,0,passed
1,2006Q2,308208,308208,0,0,passed
2,2006Q3,283975,283975,0,0,passed
3,2006Q4,302787,302787,0,0,passed
4,2007Q1,302331,302331,0,0,passed
5,2007Q2,354055,354055,0,0,passed
6,2007Q3,284980,284980,0,0,passed
7,2007Q4,296826,296826,0,0,passed
8,2008Q1,421635,421635,0,0,passed
9,2008Q2,370982,370982,0,0,passed



Total Origination loans: 11,128,442

Total Performance loans: 11,128,442

Total orphan Performance loans: 0

Total Origination loans without Performance: 0


In [13]:
import duckdb
import pandas as pd
import gc

missing_key_audit = []

for year in range(2006, 2013):
    for quarter in range(1, 5):
        vintage = f"{year}Q{quarter}"

        orig_file = ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"
        perf_file = PERF_PARQUET_DIR / f"perf_{vintage}.parquet"

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute("SET preserve_insertion_order = false")

            temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
            con.execute(f"SET temp_directory = '{temp_path}'")

            # Origination: kiểm tra loan_identifier
            orig_result = con.execute("""
                SELECT
                    COUNT(*) AS total_records,
                    COUNT(*) FILTER (
                        WHERE NULLIF(TRIM(loan_identifier), '') IS NULL
                    ) AS missing_loan_identifier
                FROM read_parquet(?)
            """, [str(orig_file)]).fetchone()

            # Performance: kiểm tra loan_identifier và period
            perf_result = con.execute("""
                SELECT
                    COUNT(*) AS total_records,
                    COUNT(*) FILTER (
                        WHERE NULLIF(TRIM(loan_identifier), '') IS NULL
                    ) AS missing_loan_identifier,
                    COUNT(*) FILTER (
                        WHERE NULLIF(TRIM(period), '') IS NULL
                    ) AS missing_period
                FROM read_parquet(?)
            """, [str(perf_file)]).fetchone()

            record = {
                "vintage": vintage,
                "origination_records": orig_result[0],
                "orig_missing_loan_id": orig_result[1],
                "performance_records": perf_result[0],
                "perf_missing_loan_id": perf_result[1],
                "perf_missing_period": perf_result[2],
            }

            missing_key_audit.append(record)

        finally:
            con.close()
            gc.collect()

        # Save progress after each vintage
        pd.DataFrame(missing_key_audit).to_csv(
            AUDIT_DIR / "missing_key_audit.csv",
            index=False
        )

        print(
            f"{vintage}: "
            f"Orig missing ID = {record['orig_missing_loan_id']:,} | "
            f"Perf missing ID = {record['perf_missing_loan_id']:,} | "
            f"Perf missing period = {record['perf_missing_period']:,}"
        )

missing_key_audit_df = pd.DataFrame(missing_key_audit)

display(missing_key_audit_df)

print("\nTOTALS")
print("Origination missing loan_identifier:",
      f"{missing_key_audit_df['orig_missing_loan_id'].sum():,}")
print("Performance missing loan_identifier:",
      f"{missing_key_audit_df['perf_missing_loan_id'].sum():,}")
print("Performance missing period:",
      f"{missing_key_audit_df['perf_missing_period'].sum():,}")

2006Q1: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2006Q2: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2006Q3: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2006Q4: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2007Q1: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2007Q2: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2007Q3: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2007Q4: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2008Q1: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2008Q2: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2008Q3: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2008Q4: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2009Q1: Orig missing ID = 0 | Perf missing ID = 0 | Perf missing period = 0
2009Q2: Orig

,vintage,origination_records,orig_missing_loan_id,performance_records,perf_missing_loan_id,perf_missing_period
0,2006Q1,298574,0,20641773,0,0
1,2006Q2,308208,0,19232256,0,0
2,2006Q3,283975,0,17091022,0,0
3,2006Q4,302787,0,19600060,0,0
4,2007Q1,302331,0,19455523,0,0
5,2007Q2,354055,0,22314866,0,0
6,2007Q3,284980,0,15847264,0,0
7,2007Q4,296826,0,16954114,0,0
8,2008Q1,421635,0,23849747,0,0
9,2008Q2,370982,0,19540165,0,0



TOTALS
Origination missing loan_identifier: 0
Performance missing loan_identifier: 0
Performance missing period: 0


In [14]:
import duckdb
import gc
from pathlib import Path

vintage = "2012Q4"

orig_file = ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"
perf_file = PERF_PARQUET_DIR / f"perf_{vintage}.parquet"

# Tạo thư mục analytical nếu chưa có
ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)
ANALYTICAL_DIR.mkdir(parents=True, exist_ok=True)

output_file = ANALYTICAL_DIR / f"loan_month_{vintage}_test.parquet"
temp_file = ANALYTICAL_DIR / f"loan_month_{vintage}_test.tmp.parquet"

# Xóa file tạm cũ nếu có
if temp_file.exists():
    temp_file.unlink()

con = duckdb.connect()

try:
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")

    temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
    con.execute(f"SET temp_directory = '{temp_path}'")

    # Đếm số dòng Performance trước join
    performance_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(perf_file)]
    ).fetchone()[0]

    # Thực hiện join và ghi ra Parquet
    orig_path_sql = str(orig_file).replace("'", "''")
    perf_path_sql = str(perf_file).replace("'", "''")
    temp_path_sql = str(temp_file).replace("'", "''")

    con.execute(f"""
        COPY (
            SELECT
                p.*,
                o.* EXCLUDE (loan_identifier)
            FROM read_parquet('{perf_path_sql}') AS p
            LEFT JOIN read_parquet('{orig_path_sql}') AS o
                ON p.loan_identifier = o.loan_identifier
        )
        TO '{temp_path_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    # Kiểm tra số dòng sau join
    joined_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(temp_file)]
    ).fetchone()[0]

    # Kiểm tra số dòng không tìm thấy thông tin Origination
    unmatched_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{temp_path_sql}')
        WHERE classic_fico IS NULL
    """).fetchone()[0]

    print(f"Performance rows before join: {performance_count:,}")
    print(f"Rows after join:              {joined_count:,}")
    print(f"Unmatched rows:               {unmatched_count:,}")
    print(f"Row count preserved:          {performance_count == joined_count}")

finally:
    con.close()
    gc.collect()

# Chỉ đổi tên file khi tất cả kiểm tra đều đạt
if (
    performance_count == joined_count
    and unmatched_count == 0
):
    if output_file.exists():
        output_file.unlink()

    temp_file.replace(output_file)
    print(f"\nJoin test PASSED. Output saved to:\n{output_file}")
else:
    print("\nJoin test FAILED. Temporary output retained for inspection.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Performance rows before join: 56,807,882
Rows after join:              56,807,882
Unmatched rows:               0
Row count preserved:          True

Join test PASSED. Output saved to:
E:\DS_project_backup\credit-risk-portfolio\processed\analytical\loan_month_2012Q4_test.parquet


In [15]:
import duckdb
import pandas as pd
import gc
from pathlib import Path

ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)
ANALYTICAL_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = AUDIT_DIR / "loan_month_integration_manifest.csv"

# Khôi phục tiến độ nếu notebook bị gián đoạn
if manifest_path.exists():
    integration_manifest = pd.read_csv(manifest_path)
    completed_vintages = set(
        integration_manifest.loc[
            integration_manifest["status"] == "completed",
            "vintage"
        ]
    )
else:
    integration_manifest = pd.DataFrame()
    completed_vintages = set()

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"

        if vintage in completed_vintages:
            print(f"{vintage}: already completed — skipped")
            continue

        orig_file = ORIG_PARQUET_DIR / f"orig_{vintage}.parquet"
        perf_file = PERF_PARQUET_DIR / f"perf_{vintage}.parquet"

        output_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"
        temp_file = ANALYTICAL_DIR / f"loan_month_{vintage}.tmp.parquet"

        if temp_file.exists():
            temp_file.unlink()

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute("SET preserve_insertion_order = false")

            temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
            con.execute(f"SET temp_directory = '{temp_path}'")

            # Count input Performance rows
            performance_count = con.execute(
                "SELECT COUNT(*) FROM read_parquet(?)",
                [str(perf_file)]
            ).fetchone()[0]

            orig_sql = str(orig_file).replace("'", "''")
            perf_sql = str(perf_file).replace("'", "''")
            temp_sql = str(temp_file).replace("'", "''")

            # Join and write to temporary Parquet
            con.execute(f"""
                COPY (
                    SELECT
                        p.*,
                        o.* EXCLUDE (loan_identifier)
                    FROM read_parquet('{perf_sql}') AS p
                    LEFT JOIN read_parquet('{orig_sql}') AS o
                        ON p.loan_identifier = o.loan_identifier
                )
                TO '{temp_sql}'
                (FORMAT PARQUET, COMPRESSION ZSTD)
            """)

            # Validate output row count
            joined_count = con.execute(
                "SELECT COUNT(*) FROM read_parquet(?)",
                [str(temp_file)]
            ).fetchone()[0]

            if performance_count != joined_count:
                raise ValueError(
                    f"{vintage}: row count mismatch "
                    f"({performance_count:,} vs {joined_count:,})"
                )

            # Replace final file only after validation
            if output_file.exists():
                output_file.unlink()

            temp_file.replace(output_file)

            record = {
                "vintage": vintage,
                "performance_records": performance_count,
                "joined_records": joined_count,
                "row_count_preserved": True,
                "output_size_gb": output_file.stat().st_size / (1024**3),
                "status": "completed"
            }

            print(
                f"{vintage}: completed | "
                f"{joined_count:,} rows | "
                f"{record['output_size_gb']:.3f} GB"
            )

        except Exception as e:
            record = {
                "vintage": vintage,
                "performance_records": None,
                "joined_records": None,
                "row_count_preserved": False,
                "output_size_gb": None,
                "status": f"failed: {str(e)}"
            }

            print(f"{vintage}: FAILED — {e}")

            if temp_file.exists():
                temp_file.unlink()

        finally:
            con.close()
            gc.collect()

        # Save progress after every vintage
        new_row = pd.DataFrame([record])

        integration_manifest = pd.concat(
            [integration_manifest, new_row],
            ignore_index=True
        )

        integration_manifest.to_csv(
            manifest_path,
            index=False
        )

        if record["status"] != "completed":
            print("Stopping batch because this vintage failed.")
            break

    else:
        continue

    break

display(integration_manifest)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q1: completed | 20,641,773 rows | 0.197 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q2: completed | 19,232,256 rows | 0.185 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q3: completed | 17,091,022 rows | 0.166 GB


... (47 outputs left out)

2012Q3: completed | 52,731,575 rows | 0.508 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012Q4: completed | 56,807,882 rows | 0.548 GB


,vintage,performance_records,joined_records,row_count_preserved,output_size_gb,status
0,2006Q1,20641773,20641773,True,0.197493,completed
1,2006Q2,19232256,19232256,True,0.184862,completed
2,2006Q3,17091022,17091022,True,0.165801,completed
3,2006Q4,19600060,19600060,True,0.191400,completed
4,2007Q1,19455523,19455523,True,0.189783,completed
5,2007Q2,22314866,22314866,True,0.217946,completed
6,2007Q3,15847264,15847264,True,0.156355,completed
7,2007Q4,16954114,16954114,True,0.168221,completed
8,2008Q1,23849747,23849747,True,0.232328,completed
9,2008Q2,19540165,19540165,True,0.188137,completed


In [16]:
import pandas as pd
from pathlib import Path

manifest_path = AUDIT_DIR / "loan_month_integration_manifest.csv"

manifest = pd.read_csv(manifest_path)

# Chỉ tính các bản tích hợp chính thức
completed = manifest[manifest["status"] == "completed"].copy()

# Kiểm tra số file thực tế
parquet_files = list(ANALYTICAL_DIR.glob("loan_month_*.parquet"))

# Loại file test khỏi danh sách chính thức
parquet_files = [
    f for f in parquet_files
    if not f.name.endswith("_test.parquet")
]

print("COMPLETED VINTAGES:", len(completed))
print("OFFICIAL PARQUET FILES:", len(parquet_files))

print(
    "TOTAL PERFORMANCE RECORDS:",
    f"{completed['performance_records'].sum():,}"
)

print(
    "TOTAL JOINED RECORDS:",
    f"{completed['joined_records'].sum():,}"
)

print(
    "ALL ROW COUNTS PRESERVED:",
    completed["row_count_preserved"].all()
)

print(
    "ALL STATUSES COMPLETED:",
    (completed["status"] == "completed").all()
)

print(
    "TOTAL PARQUET SIZE (GB):",
    round(
        sum(f.stat().st_size for f in parquet_files) / (1024**3),
        2
    )
)

print("\nFAILED OR INCOMPLETE VINTAGES:")
display(manifest[manifest["status"] != "completed"])

COMPLETED VINTAGES: 28
OFFICIAL PARQUET FILES: 28
TOTAL PERFORMANCE RECORDS: 769,725,294
TOTAL JOINED RECORDS: 769,725,294
ALL ROW COUNTS PRESERVED: True
ALL STATUSES COMPLETED: True
TOTAL PARQUET SIZE (GB): 7.38

FAILED OR INCOMPLETE VINTAGES:


,vintage,performance_records,joined_records,row_count_preserved,output_size_gb,status


In [17]:
import duckdb
import pandas as pd
import gc

# Classification mapping — provisional
classification_map = {
    "01": "Voluntary termination",
    "02": "Third-party sale",
    "03": "Short sale / charge-off",
    "09": "REO disposition",
    "15": "Whole loan sale",
    "16": "Reperforming loan securitization",
    "96": "Defect-related termination",
    "NULL": "No termination code recorded"
}

audit_results = []

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"
        perf_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute("SET preserve_insertion_order = false")

            temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
            con.execute(f"SET temp_directory = '{temp_path}'")

            result = con.execute("""
                SELECT
                    COALESCE(
                        NULLIF(TRIM(CAST(zero_balance_code AS VARCHAR)), ''),
                        'NULL'
                    ) AS zero_balance_code,

                    COUNT(*) AS performance_records,

                    COUNT(DISTINCT loan_identifier) AS unique_loans,

                    COUNT(*) FILTER (
                        WHERE TRY_CAST(actual_loss AS DOUBLE) IS NOT NULL
                    ) AS actual_loss_nonnull_records,

                    COUNT(*) FILTER (
                        WHERE TRY_CAST(actual_loss AS DOUBLE) IS NULL
                    ) AS actual_loss_null_records,

                    SUM(TRY_CAST(actual_loss AS DOUBLE))
                        AS total_actual_loss

                FROM read_parquet(?)

                GROUP BY 1
                ORDER BY 1
            """, [str(perf_file)]).fetchdf()

            result["vintage"] = vintage

            audit_results.append(result)

            print(f"{vintage}: audited — {len(result)} code groups")

        finally:
            con.close()
            gc.collect()

# Combine the small aggregated results
zero_balance_audit = pd.concat(
    audit_results,
    ignore_index=True
)

zero_balance_audit["classification"] = (
    zero_balance_audit["zero_balance_code"]
    .map(classification_map)
    .fillna("Other / unclassified code")
)

zero_balance_audit["actual_loss_nonnull_rate"] = (
    zero_balance_audit["actual_loss_nonnull_records"]
    / zero_balance_audit["performance_records"]
)

# Save vintage-level results
zero_balance_audit.to_csv(
    AUDIT_DIR / "zero_balance_code_audit_by_vintage.csv",
    index=False
)

display(zero_balance_audit.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q1: audited — 8 code groups


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q2: audited — 8 code groups
2006Q3: audited — 8 code groups


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006Q4: audited — 8 code groups


... (35 outputs left out)

2012Q3: audited — 8 code groups


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012Q4: audited — 8 code groups


,zero_balance_code,performance_records,unique_loans,actual_loss_nonnull_records,actual_loss_null_records,total_actual_loss,vintage,classification,actual_loss_nonnull_rate
0,01,261295,261295,0,261295,NaN,2006Q1,Voluntary termination,0.000000
1,02,3590,3590,3520,70,3.193843e+08,2006Q1,Third-party sale,0.980501
2,03,6031,6031,5836,195,5.372440e+08,2006Q1,Short sale / charge-off,0.967667
3,09,13106,13106,12235,871,1.113604e+09,2006Q1,REO disposition,0.933542
4,15,1264,1264,1201,63,1.132977e+08,2006Q1,Whole loan sale,0.950158
5,16,5855,5855,0,5855,NaN,2006Q1,Reperforming loan securitization,0.000000
6,96,973,973,0,973,NaN,2006Q1,Defect-related termination,0.000000
7,NULL,20349659,298343,0,20349659,NaN,2006Q1,No termination code recorded,0.000000
8,01,272072,272072,0,272072,NaN,2006Q2,Voluntary termination,0.000000
9,02,3200,3200,3133,67,2.834332e+08,2006Q2,Third-party sale,0.979062


In [18]:
summary = (
    zero_balance_audit
    .groupby(
        ["zero_balance_code", "classification"],
        as_index=False,
        dropna=False
    )
    .agg({
        "performance_records": "sum",
        "unique_loans": "sum",
        "actual_loss_nonnull_records": "sum",
        "actual_loss_null_records": "sum",
        "total_actual_loss": "sum"
    })
)

summary["actual_loss_nonnull_rate"] = (
    summary["actual_loss_nonnull_records"]
    / summary["performance_records"]
)

summary = summary.sort_values(
    "performance_records",
    ascending=False
)

summary.to_csv(
    AUDIT_DIR / "zero_balance_code_classification_summary.csv",
    index=False
)

display(summary)

print("\nTOTAL PERFORMANCE RECORDS:",
      f"{summary['performance_records'].sum():,}")

print("TOTAL UNIQUE LOAN COUNTS:",
      f"{summary['unique_loans'].sum():,}")

,zero_balance_code,classification,performance_records,unique_loans,actual_loss_nonnull_records,actual_loss_null_records,total_actual_loss,actual_loss_nonnull_rate
7,NULL,No termination code recorded,759404013,11121461,0,759404013,0.000000e+00,0.000000
0,01,Voluntary termination,9847984,9847984,0,9847984,0.000000e+00,0.000000
3,09,REO disposition,175271,175271,158868,16403,1.391226e+10,0.906413
5,16,Reperforming loan securitization,97387,97387,0,97387,0.000000e+00,0.000000
2,03,Short sale / charge-off,86273,86273,81434,4839,6.588285e+09,0.943911
1,02,Third-party sale,49267,49267,47759,1508,3.583937e+09,0.969391
6,96,Defect-related termination,44688,44688,0,44688,0.000000e+00,0.000000
4,15,Whole loan sale,20411,20411,19327,1084,1.675647e+09,0.946891



TOTAL PERFORMANCE RECORDS: 769,725,294
TOTAL UNIQUE LOAN COUNTS: 21,442,742


In [19]:
import duckdb
import pandas as pd

vintage = "2012Q4"
perf_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"

con = duckdb.connect()

try:
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 2")

    temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
    con.execute(f"SET temp_directory = '{temp_path}'")

    result = con.execute("""
        SELECT
            zero_balance_code,
            COUNT(*) AS performance_records,
            COUNT(DISTINCT loan_identifier) AS unique_loans,
            COUNT(*) FILTER (
                WHERE TRY_CAST(actual_loss AS DOUBLE) IS NOT NULL
            ) AS actual_loss_nonnull_records,
            COUNT(DISTINCT loan_identifier) FILTER (
                WHERE TRY_CAST(actual_loss AS DOUBLE) IS NOT NULL
            ) AS loans_with_actual_loss
        FROM read_parquet(?)
        WHERE TRIM(zero_balance_code) IN ('02', '03', '09', '15')
        GROUP BY zero_balance_code
        ORDER BY zero_balance_code
    """, [str(perf_file)]).fetchdf()

    display(result)

finally:
    con.close()

,zero_balance_code,performance_records,unique_loans,actual_loss_nonnull_records,loans_with_actual_loss
0,02,931,931,921,921
1,03,994,994,983,983
2,09,1094,1094,1074,1074
3,15,414,414,411,411


In [20]:
con = duckdb.connect()

try:
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 2")

    sample = con.execute("""
        WITH selected_loans AS (
            SELECT loan_identifier
            FROM read_parquet(?)
            WHERE TRIM(zero_balance_code) = '09'
            GROUP BY loan_identifier
            LIMIT 5
        )
        SELECT
            p.loan_identifier,
            p.period,
            p.current_actual_upb,
            p.current_loan_delinquency_status,
            p.zero_balance_code,
            p.zero_balance_effective_date,
            p.actual_loss
        FROM read_parquet(?) p
        INNER JOIN selected_loans s
            USING (loan_identifier)
        ORDER BY p.loan_identifier, p.period
    """, [str(perf_file), str(perf_file)]).fetchdf()

    display(sample)

finally:
    con.close()

,loan_identifier,period,current_actual_upb,current_loan_delinquency_status,zero_balance_code,zero_balance_effective_date,actual_loss
0,F12Q40001238,201212,105000.00,00,NaN,NaN,NaN
1,F12Q40001238,201301,104000.00,00,NaN,NaN,NaN
2,F12Q40001238,201302,104000.00,00,NaN,NaN,NaN
3,F12Q40001238,201303,104000.00,00,NaN,NaN,NaN
4,F12Q40001238,201304,104000.00,00,NaN,NaN,NaN
...,...,...,...,...,...,...,...
260,F12Q40006829,201501,146754.32,11,NaN,NaN,NaN
261,F12Q40006829,201502,146754.32,12,NaN,NaN,NaN
262,F12Q40006829,201503,146754.32,RA,NaN,NaN,NaN
263,F12Q40006829,201504,146754.32,RA,NaN,NaN,NaN


In [21]:
import duckdb
import pandas as pd

vintage = "2012Q4"
perf_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"

con = duckdb.connect()

try:
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 2")

    temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
    con.execute(f"SET temp_directory = '{temp_path}'")

    # Summary: how many loans have repeated Zero Balance Codes?
    duplicate_summary = con.execute("""
        WITH loan_code AS (
            SELECT
                loan_identifier,
                TRIM(zero_balance_code) AS zero_balance_code,
                COUNT(*) AS record_count,
                COUNT(DISTINCT period) AS distinct_periods,
                COUNT(*) FILTER (
                    WHERE TRY_CAST(actual_loss AS DOUBLE) IS NOT NULL
                ) AS loss_records,
                COUNT(DISTINCT TRY_CAST(actual_loss AS DOUBLE))
                    AS distinct_loss_values
            FROM read_parquet(?)
            WHERE TRIM(zero_balance_code) IN ('02', '03', '09', '15')
            GROUP BY 1, 2
        )
        SELECT
            zero_balance_code,
            COUNT(*) AS unique_loans,
            SUM(record_count) AS performance_records,
            COUNT(*) FILTER (
                WHERE record_count > 1
            ) AS loans_with_repeated_code,
            SUM(record_count - 1) AS excess_records,
            COUNT(*) FILTER (
                WHERE distinct_loss_values > 1
            ) AS loans_with_multiple_loss_values
        FROM loan_code
        GROUP BY 1
        ORDER BY 1
    """, [str(perf_file)]).fetchdf()

    display(duplicate_summary)

    # Inspect individual loans with repeated codes
    duplicate_examples = con.execute("""
        WITH loan_code AS (
            SELECT
                loan_identifier,
                TRIM(zero_balance_code) AS zero_balance_code,
                COUNT(*) AS record_count
            FROM read_parquet(?)
            WHERE TRIM(zero_balance_code) IN ('02', '03', '09', '15')
            GROUP BY 1, 2
            HAVING COUNT(*) > 1
        )
        SELECT
            p.loan_identifier,
            p.period,
            p.zero_balance_code,
            p.zero_balance_effective_date,
            p.current_actual_upb,
            p.actual_loss
        FROM read_parquet(?) p
        INNER JOIN loan_code d
            ON p.loan_identifier = d.loan_identifier
            AND TRIM(p.zero_balance_code) = d.zero_balance_code
        ORDER BY p.loan_identifier, p.period
        LIMIT 100
    """, [str(perf_file), str(perf_file)]).fetchdf()

    display(duplicate_examples)

finally:
    con.close()

,zero_balance_code,unique_loans,performance_records,loans_with_repeated_code,excess_records,loans_with_multiple_loss_values
0,02,931,931.0,0,0.0,0
1,03,994,994.0,0,0.0,0
2,09,1094,1094.0,0,0.0,0
3,15,414,414.0,0,0.0,0


,loan_identifier,period,zero_balance_code,zero_balance_effective_date,current_actual_upb,actual_loss


In [22]:
import duckdb
import pandas as pd
import gc

event_audit_results = []

event_codes = ("02", "03", "09", "15")

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"
        perf_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute("SET preserve_insertion_order = false")

            temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
            con.execute(f"SET temp_directory = '{temp_path}'")

            result = con.execute("""
                WITH event_records AS (
                    SELECT
                        loan_identifier,
                        TRIM(zero_balance_code) AS zero_balance_code,
                        actual_loss
                    FROM read_parquet(?)
                    WHERE TRIM(zero_balance_code)
                        IN ('02', '03', '09', '15')
                ),

                loan_events AS (
                    SELECT
                        loan_identifier,
                        zero_balance_code,
                        COUNT(*) AS event_record_count,
                        COUNT(*) FILTER (
                            WHERE TRY_CAST(actual_loss AS DOUBLE) IS NOT NULL
                        ) AS loss_record_count
                    FROM event_records
                    GROUP BY loan_identifier, zero_balance_code
                )

                SELECT
                    zero_balance_code,
                    COUNT(*) AS unique_loans,
                    SUM(event_record_count) AS event_records,

                    COUNT(*) FILTER (
                        WHERE event_record_count > 1
                    ) AS loans_with_repeated_events,

                    COUNT(*) FILTER (
                        WHERE loss_record_count > 0
                    ) AS loans_with_actual_loss,

                    COUNT(*) FILTER (
                        WHERE loss_record_count = 0
                    ) AS loans_without_actual_loss

                FROM loan_events
                GROUP BY zero_balance_code
                ORDER BY zero_balance_code

            """, [str(perf_file)]).fetchdf()

            result["vintage"] = vintage
            event_audit_results.append(result)

            print(f"{vintage}: completed")

        finally:
            con.close()
            gc.collect()

event_audit = pd.concat(
    event_audit_results,
    ignore_index=True
)

event_audit.to_csv(
    AUDIT_DIR / "default_event_uniqueness_audit.csv",
    index=False
)

display(event_audit)

2006Q1: completed
2006Q2: completed
2006Q3: completed
2006Q4: completed
2007Q1: completed
2007Q2: completed
2007Q3: completed
2007Q4: completed
2008Q1: completed
2008Q2: completed
2008Q3: completed
2008Q4: completed
2009Q1: completed


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009Q2: completed
2009Q3: completed
2009Q4: completed
2010Q1: completed
2010Q2: completed
2010Q3: completed
2010Q4: completed
2011Q1: completed
2011Q2: completed
2011Q3: completed
2011Q4: completed
2012Q1: completed
2012Q2: completed
2012Q3: completed


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012Q4: completed


,zero_balance_code,unique_loans,event_records,loans_with_repeated_events,loans_with_actual_loss,loans_without_actual_loss,vintage
0,02,3590,3590.0,0,3520,70,2006Q1
1,03,6031,6031.0,0,5836,195,2006Q1
2,09,13106,13106.0,0,12235,871,2006Q1
3,15,1264,1264.0,0,1201,63,2006Q1
4,02,3200,3200.0,0,3133,67,2006Q2
...,...,...,...,...,...,...,...
107,15,378,378.0,0,374,4,2012Q3
108,02,931,931.0,0,921,10,2012Q4
109,03,994,994.0,0,983,11,2012Q4
110,09,1094,1094.0,0,1074,20,2012Q4


In [23]:
event_summary = (
    event_audit
    .groupby("zero_balance_code", as_index=False)
    .agg({
        "unique_loans": "sum",
        "event_records": "sum",
        "loans_with_repeated_events": "sum",
        "loans_with_actual_loss": "sum",
        "loans_without_actual_loss": "sum"
    })
)

event_summary["event_uniqueness_pass"] = (
    event_summary["unique_loans"]
    == event_summary["event_records"]
)

display(event_summary)

print(
    "TOTAL UNIQUE EVENT LOANS:",
    f"{event_summary['unique_loans'].sum():,}"
)

print(
    "TOTAL EVENT RECORDS:",
    f"{event_summary['event_records'].sum():,}"
)

print(
    "TOTAL REPEATED EVENT LOANS:",
    f"{event_summary['loans_with_repeated_events'].sum():,}"
)

,zero_balance_code,unique_loans,event_records,loans_with_repeated_events,loans_with_actual_loss,loans_without_actual_loss,event_uniqueness_pass
0,02,49267,49267.0,0,47759,1508,True
1,03,86273,86273.0,0,81434,4839,True
2,09,175271,175271.0,0,158868,16403,True
3,15,20411,20411.0,0,19327,1084,True


TOTAL UNIQUE EVENT LOANS: 331,222
TOTAL EVENT RECORDS: 331,222.0
TOTAL REPEATED EVENT LOANS: 0


In [24]:
import duckdb
import pandas as pd
import gc
from pathlib import Path

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)
EVENT_DIR.mkdir(parents=True, exist_ok=True)

event_files = []

for year in range(2006, 2013):
    for quarter in range(1, 5):

        vintage = f"{year}Q{quarter}"
        perf_file = ANALYTICAL_DIR / f"loan_month_{vintage}.parquet"

        output_file = EVENT_DIR / f"credit_events_{vintage}.parquet"
        temp_file = EVENT_DIR / f"credit_events_{vintage}.tmp.parquet"

        if temp_file.exists():
            temp_file.unlink()

        con = duckdb.connect()

        try:
            con.execute("SET memory_limit = '3GB'")
            con.execute("SET threads = 2")
            con.execute("SET preserve_insertion_order = false")

            temp_path = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
            con.execute(f"SET temp_directory = '{temp_path}'")

            perf_sql = str(perf_file).replace("'", "''")
            temp_sql = str(temp_file).replace("'", "''")

            con.execute(f"""
                COPY (
                    SELECT
                        loan_identifier,
                        period,
                        zero_balance_code,
                        zero_balance_effective_date,
                        current_actual_upb,
                        zero_balance_removal_upb,
                        actual_loss,
                        delinquent_accrued_interest,
                        net_sales_proceeds,
                        total_expenses,
                        mi_recoveries,
                        non_mi_recoveries,
                        underwriting_defect_and_major_servicing_defect_settlement_date,

                        CASE
                            WHEN TRIM(zero_balance_code)
                                IN ('02', '03', '09')
                            THEN 1 ELSE 0
                        END AS strict_credit_event,

                        CASE
                            WHEN TRIM(zero_balance_code)
                                IN ('02', '03', '09', '15')
                            THEN 1 ELSE 0
                        END AS extended_credit_event

                    FROM read_parquet('{perf_sql}')

                    WHERE TRIM(zero_balance_code)
                        IN ('02', '03', '09', '15')

                )
                TO '{temp_sql}'
                (FORMAT PARQUET, COMPRESSION ZSTD)
            """)

            count = con.execute(
                "SELECT COUNT(*) FROM read_parquet(?)",
                [str(temp_file)]
            ).fetchone()[0]

            if output_file.exists():
                output_file.unlink()

            temp_file.replace(output_file)

            event_files.append({
                "vintage": vintage,
                "event_records": count,
                "status": "completed",
                "file_size_mb": round(
                    output_file.stat().st_size / (1024**2), 2
                )
            })

            print(f"{vintage}: {count:,} event records")

        finally:
            con.close()
            gc.collect()

event_manifest = pd.DataFrame(event_files)

event_manifest.to_csv(
    AUDIT_DIR / "credit_event_dataset_manifest.csv",
    index=False
)

display(event_manifest)

print(
    "\nTOTAL EVENT RECORDS:",
    f"{event_manifest['event_records'].sum():,}"
)

2006Q1: 23,991 event records
2006Q2: 23,560 event records
2006Q3: 22,500 event records
2006Q4: 26,082 event records
2007Q1: 26,485 event records
2007Q2: 32,330 event records
2007Q3: 26,100 event records
2007Q4: 27,356 event records
2008Q1: 26,095 event records
2008Q2: 17,866 event records
2008Q3: 8,936 event records
2008Q4: 5,823 event records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009Q1: 5,483 event records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009Q2: 6,328 event records
2009Q3: 5,748 event records
2009Q4: 5,289 event records
2010Q1: 4,629 event records
2010Q2: 4,045 event records
2010Q3: 4,226 event records
2010Q4: 4,135 event records
2011Q1: 3,225 event records
2011Q2: 2,402 event records
2011Q3: 2,043 event records
2011Q4: 1,942 event records
2012Q1: 2,729 event records
2012Q2: 4,719 event records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012Q3: 3,722 event records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012Q4: 3,433 event records


,vintage,event_records,status,file_size_mb
0,2006Q1,23991,completed,0.77
1,2006Q2,23560,completed,0.76
2,2006Q3,22500,completed,0.73
3,2006Q4,26082,completed,0.82
4,2007Q1,26485,completed,0.83
5,2007Q2,32330,completed,1.01
6,2007Q3,26100,completed,0.82
7,2007Q4,27356,completed,0.86
8,2008Q1,26095,completed,0.83
9,2008Q2,17866,completed,0.58



TOTAL EVENT RECORDS: 331,222


In [25]:
from pathlib import Path
import duckdb
import pandas as pd

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

try:
    event_files = sorted(EVENT_DIR.glob("credit_events_*.parquet"))

    # 1. Overall event-level integrity
    integrity = con.execute("""
        SELECT
            COUNT(*) AS event_records,
            COUNT(DISTINCT loan_identifier) AS unique_loans,
            COUNT(*) - COUNT(DISTINCT loan_identifier)
                AS duplicate_excess,
            COUNT(*) FILTER (
                WHERE zero_balance_effective_date IS NULL
                   OR TRIM(zero_balance_effective_date) = ''
            ) AS missing_effective_date,
            COUNT(*) FILTER (
                WHERE period IS NULL OR TRIM(period) = ''
            ) AS missing_period
        FROM read_parquet(?)
    """, [ [str(p) for p in event_files] ]).df()

    display(integrity)

    # 2. Compare reporting period and effective date
    date_comparison = con.execute("""
        SELECT
            COUNT(*) AS total_events,
            COUNT(*) FILTER (
                WHERE TRIM(period) =
                      TRIM(zero_balance_effective_date)
            ) AS same_period,
            COUNT(*) FILTER (
                WHERE TRIM(period) <>
                      TRIM(zero_balance_effective_date)
            ) AS different_period,
            COUNT(*) FILTER (
                WHERE period IS NULL
                   OR zero_balance_effective_date IS NULL
            ) AS missing_dates
        FROM read_parquet(?)
    """, [[str(p) for p in event_files]]).df()

    display(date_comparison)

    # 3. Inspect examples where dates differ
    examples = con.execute("""
        SELECT
            loan_identifier,
            zero_balance_code,
            period,
            zero_balance_effective_date,
            actual_loss
        FROM read_parquet(?)
        WHERE period IS NOT NULL
          AND zero_balance_effective_date IS NOT NULL
          AND TRIM(period) <>
              TRIM(zero_balance_effective_date)
        LIMIT 20
    """, [[str(p) for p in event_files]]).df()

    display(examples)

finally:
    con.close()

,event_records,unique_loans,duplicate_excess,missing_effective_date,missing_period
0,331222,331222,0,0,0


,total_events,same_period,different_period,missing_dates
0,331222,329010,2212,0


,loan_identifier,zero_balance_code,period,zero_balance_effective_date,actual_loss
0,F06Q10007222,09,202601,202512,140494.05
1,F06Q10008182,03,201003,201009,52122.72
2,F06Q10023315,15,201510,201511,156900.97
3,F06Q10023958,15,201510,201511,187528.47
4,F06Q10038190,15,201511,201512,67513.50
5,F06Q10042232,15,201510,201511,242964.10
6,F06Q10052463,15,201510,201511,73919.89
7,F06Q10052645,03,201007,201009,2561.57
8,F06Q10057779,03,201004,201007,99438.47
9,F06Q10069349,03,200912,201001,47813.36


In [26]:
import duckdb
import pandas as pd
from pathlib import Path

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

event_files = sorted(EVENT_DIR.glob("credit_events_*.parquet"))

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")

try:
    # 1. Discrepancy by event code
    by_code = con.execute("""
        SELECT
            TRIM(zero_balance_code) AS event_code,
            COUNT(*) AS total_events,
            COUNT(*) FILTER (
                WHERE TRIM(period) <>
                      TRIM(zero_balance_effective_date)
            ) AS different_period,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE TRIM(period) <>
                          TRIM(zero_balance_effective_date)
                ) / COUNT(*),
                3
            ) AS discrepancy_rate_pct
        FROM read_parquet(?)
        GROUP BY 1
        ORDER BY 1
    """, [[str(p) for p in event_files]]).df()

    display(by_code)

    # 2. Month difference between reporting period
    # and effective date
    lag_distribution = con.execute("""
        WITH dates AS (
            SELECT
                TRIM(zero_balance_code) AS event_code,
                CAST(SUBSTR(period, 1, 4) AS INTEGER) AS period_year,
                CAST(SUBSTR(period, 5, 2) AS INTEGER) AS period_month,
                CAST(
                    SUBSTR(zero_balance_effective_date, 1, 4)
                    AS INTEGER
                ) AS effective_year,
                CAST(
                    SUBSTR(zero_balance_effective_date, 5, 2)
                    AS INTEGER
                ) AS effective_month
            FROM read_parquet(?)
            WHERE period IS NOT NULL
              AND zero_balance_effective_date IS NOT NULL
        ),
        calculated AS (
            SELECT
                event_code,
                (period_year * 12 + period_month)
                - (effective_year * 12 + effective_month)
                    AS reporting_lag_months
            FROM dates
        )
        SELECT
            event_code,
            reporting_lag_months,
            COUNT(*) AS event_count
        FROM calculated
        WHERE reporting_lag_months <> 0
        GROUP BY 1, 2
        ORDER BY 1, 2
    """, [[str(p) for p in event_files]]).df()

    display(lag_distribution)

    # 3. Discrepancy by vintage
    by_vintage = con.execute("""
        SELECT
            REGEXP_EXTRACT(
                filename,
                'credit_events_(.*)\\.parquet',
                1
            ) AS vintage,
            COUNT(*) AS total_events,
            COUNT(*) FILTER (
                WHERE TRIM(period) <>
                      TRIM(zero_balance_effective_date)
            ) AS different_period
        FROM read_parquet(?, filename = true)
        GROUP BY 1
        ORDER BY 1
    """, [[str(p) for p in event_files]]).df()

    display(by_vintage)

finally:
    con.close()

,event_code,total_events,different_period,discrepancy_rate_pct
0,02,49267,22,0.045
1,03,86273,1085,1.258
2,09,175271,9,0.005
3,15,20411,1096,5.370


,event_code,reporting_lag_months,event_count
0,02,-1,22
1,03,-183,1
2,03,-84,1
3,03,-69,1
4,03,-46,1
5,03,-37,1
6,03,-33,1
7,03,-30,1
8,03,-26,2
9,03,-25,2


,vintage,total_events,different_period
0,2006Q1,23991,143
1,2006Q2,23560,132
2,2006Q3,22500,132
3,2006Q4,26082,124
4,2007Q1,26485,132
5,2007Q2,32330,168
6,2007Q3,26100,167
7,2007Q4,27356,201
8,2008Q1,26095,192
9,2008Q2,17866,158


In [27]:
import duckdb
import pandas as pd
from pathlib import Path

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)

event_files = sorted(EVENT_DIR.glob("credit_events_*.parquet"))

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

try:
    # 1. Find extreme Code 03 discrepancies
    extreme_loans = con.execute("""
        WITH events AS (
            SELECT
                loan_identifier,
                TRIM(zero_balance_code) AS event_code,
                period,
                zero_balance_effective_date,
                CAST(SUBSTR(period, 1, 4) AS INTEGER) * 12
                    + CAST(SUBSTR(period, 5, 2) AS INTEGER)
                    AS period_month,
                CAST(
                    SUBSTR(zero_balance_effective_date, 1, 4)
                    AS INTEGER
                ) * 12
                    + CAST(
                        SUBSTR(zero_balance_effective_date, 5, 2)
                        AS INTEGER
                    ) AS effective_month
            FROM read_parquet(?)
            WHERE TRIM(zero_balance_code) = '03'
        )
        SELECT
            loan_identifier,
            event_code,
            period,
            zero_balance_effective_date,
            period_month - effective_month AS reporting_lag_months
        FROM events
        WHERE period_month <> effective_month
        ORDER BY reporting_lag_months ASC
        LIMIT 20
    """, [[str(p) for p in event_files]]).df()

    display(extreme_loans)

    # 2. Inspect their complete Performance histories
    loan_ids = extreme_loans["loan_identifier"].tolist()

    history = con.execute("""
        SELECT
            loan_identifier,
            period,
            current_loan_delinquency_status,
            current_actual_upb,
            zero_balance_code,
            zero_balance_effective_date,
            actual_loss
        FROM read_parquet(?)
        WHERE loan_identifier IN (
            SELECT UNNEST(?)
        )
        ORDER BY loan_identifier, period
    """, [
        [str(p) for p in ANALYTICAL_DIR.glob("loan_month_*.parquet")
         if not p.name.endswith("_test.parquet")],
        loan_ids
    ]).df()

    display(history)

finally:
    con.close()

,loan_identifier,event_code,period,zero_balance_effective_date,reporting_lag_months
0,F06Q40308980,03,200701,202204,-183
1,F07Q10438925,03,201008,201708,-84
2,F07Q10409451,03,201008,201605,-69
3,F09Q20239857,03,201205,201603,-46
4,F07Q10093421,03,201202,201503,-37
5,F08Q10443703,03,201408,201705,-33
6,F09Q20428633,03,201910,202204,-30
7,F06Q10366578,03,201503,201705,-26
8,F08Q20089903,03,200807,201009,-26
9,F07Q40070887,03,201307,201508,-25


,loan_identifier,period,current_loan_delinquency_status,current_actual_upb,zero_balance_code,zero_balance_effective_date,actual_loss
0,F06Q10366578,200705,00,148482.67,NaN,NaN,NaN
1,F06Q10366578,200706,00,148345.33,NaN,NaN,NaN
2,F06Q10366578,200707,00,148207.21,NaN,NaN,NaN
3,F06Q10366578,200708,00,148068.29,NaN,NaN,NaN
4,F06Q10366578,200709,00,147928.58,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1124,F10Q20219653,201412,38,75906.04,NaN,NaN,NaN
1125,F10Q20219653,201501,39,75906.04,NaN,NaN,NaN
1126,F10Q20219653,201502,40,75906.04,NaN,NaN,NaN
1127,F10Q20219653,201503,41,75906.04,NaN,NaN,NaN


In [28]:
import duckdb
import pandas as pd
from pathlib import Path

ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

event_files = sorted(EVENT_DIR.glob("credit_events_*.parquet"))

# Five largest discrepancies + one previously inspected example
selected_loans = [
    "F06Q40308980",
    "F07Q10438925",
    "F07Q10409451",
    "F09Q20239857",
    "F07Q10093421",
    "F10Q20219653"
]

perf_files = [
    str(p) for p in ANALYTICAL_DIR.glob("loan_month_*.parquet")
    if not p.name.endswith("_test.parquet")
]

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

try:
    # 1. Full monthly history
    history = con.execute("""
        SELECT
            loan_identifier,
            period,
            current_loan_delinquency_status,
            TRY_CAST(current_actual_upb AS DOUBLE) AS current_actual_upb,
            zero_balance_code,
            zero_balance_effective_date,
            TRY_CAST(actual_loss AS DOUBLE) AS actual_loss
        FROM read_parquet(?)
        WHERE loan_identifier IN (SELECT UNNEST(?))
        ORDER BY loan_identifier, period
    """, [perf_files, selected_loans]).df()

    display(history)

    # 2. Loan-level timeline summary
    summary = con.execute("""
        WITH h AS (
            SELECT
                loan_identifier,
                period,
                current_loan_delinquency_status,
                TRY_CAST(current_actual_upb AS DOUBLE) AS upb,
                zero_balance_code,
                zero_balance_effective_date
            FROM read_parquet(?)
            WHERE loan_identifier IN (SELECT UNNEST(?))
        ),
        classified AS (
            SELECT
                *,
                CASE
                    WHEN TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 3
                    THEN 1 ELSE 0
                END AS is_90plus
            FROM h
        )
        SELECT
            loan_identifier,
            COUNT(*) AS performance_records,
            MIN(period) AS first_period,
            MAX(period) AS last_period,

            MIN(period) FILTER (
                WHERE is_90plus = 1
            ) AS first_90plus_period,

            MIN(period) FILTER (
                WHERE TRIM(current_loan_delinquency_status) = 'RA'
            ) AS first_reo_period,

            MIN(period) FILTER (
                WHERE upb = 0
            ) AS first_zero_upb_period,

            MAX(period) FILTER (
                WHERE TRIM(zero_balance_code) IN ('02','03','09','15')
            ) AS event_reporting_period,

            MAX(zero_balance_effective_date)
                AS effective_date,

            MAX(zero_balance_code) FILTER (
                WHERE TRIM(zero_balance_code) IN ('02','03','09','15')
            ) AS event_code

        FROM classified
        GROUP BY loan_identifier
        ORDER BY loan_identifier
    """, [perf_files, selected_loans]).df()

    display(summary)

finally:
    con.close()

,loan_identifier,period,current_loan_delinquency_status,current_actual_upb,zero_balance_code,zero_balance_effective_date,actual_loss
0,F06Q40308980,200701,00,0.00,03,202204,65740.63
1,F07Q10093421,200702,00,56000.00,NaN,NaN,NaN
2,F07Q10093421,200703,00,56000.00,NaN,NaN,NaN
3,F07Q10093421,200704,00,56000.00,NaN,NaN,NaN
4,F07Q10093421,200705,00,56000.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...
216,F10Q20219653,201412,38,75906.04,NaN,NaN,NaN
217,F10Q20219653,201501,39,75906.04,NaN,NaN,NaN
218,F10Q20219653,201502,40,75906.04,NaN,NaN,NaN
219,F10Q20219653,201503,41,75906.04,NaN,NaN,NaN


,loan_identifier,performance_records,first_period,last_period,first_90plus_period,first_reo_period,first_zero_upb_period,event_reporting_period,effective_date,event_code
0,F06Q40308980,1,200701,200701,NaN,None,200701,200701,202204,03
1,F07Q10093421,61,200702,201202,200911,None,201202,201202,201503,03
2,F07Q10409451,37,200708,201008,200812,None,201008,201008,201605,03
3,F07Q10438925,28,200805,201008,200904,None,201008,201008,201708,03
4,F09Q20239857,36,200906,201205,201202,None,201205,201205,201603,03
5,F10Q20219653,58,201007,201504,201112,None,201504,201504,201705,03


In [29]:
import duckdb
import pandas as pd
from pathlib import Path

EVENT_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\events"
)

ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)

event_files = sorted(EVENT_DIR.glob("credit_events_*.parquet"))

perf_files = [
    str(p) for p in ANALYTICAL_DIR.glob("loan_month_*.parquet")
    if not p.name.endswith("_test.parquet")
]

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

try:
    coverage = con.execute("""
        WITH event_loans AS (
            SELECT
                loan_identifier,
                TRIM(zero_balance_code) AS event_code,
                period AS event_reporting_period,
                zero_balance_effective_date
            FROM read_parquet(?)
        ),

        loan_history AS (
            SELECT
                loan_identifier,
                COUNT(*) AS performance_records,

                MIN(period) AS first_period,
                MAX(period) AS last_period,

                COUNT(*) FILTER (
                    WHERE TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 3
                ) AS months_90plus,

                MIN(period) FILTER (
                    WHERE TRY_CAST(
                        TRIM(current_loan_delinquency_status)
                        AS INTEGER
                    ) >= 3
                ) AS first_90plus_period,

                COUNT(*) FILTER (
                    WHERE TRIM(current_loan_delinquency_status) = 'RA'
                ) AS months_REO,

                MIN(period) FILTER (
                    WHERE TRIM(current_loan_delinquency_status) = 'RA'
                ) AS first_REO_period,

                COUNT(*) FILTER (
                    WHERE TRY_CAST(current_actual_upb AS DOUBLE) = 0
                ) AS zero_UPB_records,

                MIN(period) FILTER (
                    WHERE TRY_CAST(current_actual_upb AS DOUBLE) = 0
                ) AS first_zero_UPB_period

            FROM read_parquet(?)
            GROUP BY loan_identifier
        )

        SELECT
            e.*,
            h.performance_records,
            h.first_period,
            h.last_period,
            h.months_90plus,
            h.first_90plus_period,
            h.months_REO,
            h.first_REO_period,
            h.zero_UPB_records,
            h.first_zero_UPB_period,

            CASE
                WHEN h.months_90plus > 0 THEN 1 ELSE 0
            END AS has_90plus,

            CASE
                WHEN h.months_REO > 0 THEN 1 ELSE 0
            END AS has_REO,

            CASE
                WHEN h.zero_UPB_records > 0 THEN 1 ELSE 0
            END AS has_zero_UPB,

            CASE
                WHEN h.last_period = e.event_reporting_period
                THEN 1 ELSE 0
            END AS last_period_matches_event,

            CASE
                WHEN e.zero_balance_effective_date > h.last_period
                THEN 1 ELSE 0
            END AS effective_date_after_last_period

        FROM event_loans e
        LEFT JOIN loan_history h
            USING (loan_identifier)
    """, [
        [str(p) for p in event_files],
        perf_files
    ]).df()

    display(coverage.head(10))

    # Overall coverage summary
    summary = pd.DataFrame({
        "metric": [
            "Total event loans",
            "Loans with 90+ DPD",
            "Loans with REO status",
            "Loans with zero UPB",
            "Last period matches event reporting period",
            "Effective date after last observed period",
            "Loans without 90+ DPD history"
        ],
        "count": [
            len(coverage),
            coverage["has_90plus"].sum(),
            coverage["has_REO"].sum(),
            coverage["has_zero_UPB"].sum(),
            coverage["last_period_matches_event"].sum(),
            coverage["effective_date_after_last_period"].sum(),
            (coverage["has_90plus"] == 0).sum()
        ]
    })

    summary["percentage"] = (
        summary["count"] / len(coverage) * 100
    ).round(2)

    display(summary)

finally:
    con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,loan_identifier,event_code,event_reporting_period,zero_balance_effective_date,performance_records,first_period,last_period,months_90plus,first_90plus_period,months_REO,first_REO_period,zero_UPB_records,first_zero_UPB_period,has_90plus,has_REO,has_zero_UPB,last_period_matches_event,effective_date_after_last_period
0,F06Q10010878,03,201111,201111,70,200602,201111,4,201108,0,NaN,1,201111,1,0,1,1,0
1,F06Q10015662,09,200810,200810,33,200602,200810,5,200802,4,200807,1,200810,1,1,1,1,0
2,F06Q10022308,15,201608,201608,127,200602,201608,72,201002,0,NaN,1,201608,1,0,1,1,0
3,F06Q10025729,09,201404,201404,99,200602,201404,31,201010,12,201305,1,201404,1,1,1,1,0
4,F06Q10046784,09,201508,201508,115,200602,201508,58,201002,9,201412,1,201508,1,1,1,1,0
5,F06Q10055811,09,200908,200908,42,200603,200908,9,200806,6,200903,1,200908,1,1,1,1,0
6,F06Q10068068,09,201412,201412,106,200603,201412,27,201103,13,201312,1,201412,1,1,1,1,0
7,F06Q10081860,09,201007,201007,54,200602,201007,16,200703,7,201001,1,201007,1,1,1,1,0
8,F06Q10091316,02,201206,201206,76,200603,201206,38,200812,0,NaN,1,201206,1,0,1,1,0
9,F06Q10095100,09,201102,201102,60,200603,201102,9,200908,10,201005,1,201102,1,1,1,1,0


,metric,count,percentage
0,Total event loans,331222,100.00
1,Loans with 90+ DPD,328027,99.04
2,Loans with REO status,175274,52.92
3,Loans with zero UPB,331222,100.00
4,Last period matches event reporting period,331222,100.00
5,Effective date after last observed period,2204,0.67
6,Loans without 90+ DPD history,3195,0.96


In [31]:
# 1. Cross-tabulation: event code x 90+ DPD history

event_code_summary = (
    coverage
    .groupby("event_code", dropna=False)
    .agg(
        total_loans=("loan_identifier", "nunique"),
        loans_with_90plus=("has_90plus", "sum"),
        loans_without_90plus=(
            "has_90plus",
            lambda x: (x == 0).sum()
        ),
        loans_with_REO=("has_REO", "sum"),
        loans_with_zero_UPB=("has_zero_UPB", "sum")
    )
    .reset_index()
)

event_code_summary["pct_with_90plus"] = (
    event_code_summary["loans_with_90plus"]
    / event_code_summary["total_loans"] * 100
).round(2)

display(event_code_summary)


# 2. Inspect loans without 90+ DPD history

no_90plus = coverage.loc[
    coverage["has_90plus"] == 0
].copy()

display(no_90plus.head(30))


# 3. Check available columns
print("Available columns:")
print(no_90plus.columns.tolist())

# 4. Observation length, only if the column exists
if "performance_records" in no_90plus.columns:
    display(no_90plus["performance_records"].describe())
else:
    print(
        "performance_records is unavailable in coverage. "
        "Observation length analysis will be performed separately."
    )

,event_code,total_loans,loans_with_90plus,loans_without_90plus,loans_with_REO,loans_with_zero_UPB,pct_with_90plus
0,02,49267,49267,0,8,49267,100.00
1,03,86273,83357,2916,3,86273,96.62
2,09,175271,174992,279,175263,175271,99.84
3,15,20411,20411,0,0,20411,100.00


,loan_identifier,event_code,event_reporting_period,zero_balance_effective_date,performance_records,first_period,last_period,months_90plus,first_90plus_period,months_REO,first_REO_period,zero_UPB_records,first_zero_UPB_period,has_90plus,has_REO,has_zero_UPB,last_period_matches_event,effective_date_after_last_period
263,F07Q20406670,03,201104,201104,46,200707,201104,0,NaN,0,NaN,1,201104,0,0,1,1,0
280,F07Q30054413,09,200902,200902,19,200708,200902,0,NaN,5,200810,1,200902,0,1,1,1,0
444,F08Q20028445,03,201307,201307,63,200805,201307,0,NaN,0,NaN,1,201307,0,0,1,1,0
692,F07Q20073521,03,201207,201207,63,200705,201207,0,NaN,0,NaN,1,201207,0,0,1,1,0
838,F07Q40322007,03,200811,200811,12,200712,200811,0,NaN,0,NaN,1,200811,0,0,1,1,0
1178,F07Q10427971,03,201209,201209,58,200712,201209,0,NaN,0,NaN,1,201209,0,0,1,1,0
1187,F07Q20038225,03,201108,201108,53,200704,201108,0,NaN,0,NaN,1,201108,0,0,1,1,0
1219,F07Q20315226,03,201202,201202,57,200706,201202,0,NaN,0,NaN,1,201202,0,0,1,1,0
1276,F07Q30175814,03,201302,201302,67,200708,201302,0,NaN,0,NaN,1,201302,0,0,1,1,0
1306,F07Q30414770,03,201306,201306,69,200710,201306,0,NaN,0,NaN,1,201306,0,0,1,1,0


Available columns:
['loan_identifier', 'event_code', 'event_reporting_period', 'zero_balance_effective_date', 'performance_records', 'first_period', 'last_period', 'months_90plus', 'first_90plus_period', 'months_REO', 'first_REO_period', 'zero_UPB_records', 'first_zero_UPB_period', 'has_90plus', 'has_REO', 'has_zero_UPB', 'last_period_matches_event', 'effective_date_after_last_period']


count    3195.000000
mean       45.460407
std        22.155127
min         1.000000
25%        29.000000
50%        43.000000
75%        58.000000
max       191.000000
Name: performance_records, dtype: float64

In [32]:

import duckdb
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

ANALYTICAL_DIR = Path(
    r"E:\DS_project_backup\credit-risk-portfolio\processed\analytical"
)

# Get the 3,195 loans without 90+ DPD history
no_90plus = coverage.loc[
    coverage["has_90plus"] == 0
].copy()

loan_ids = no_90plus["loan_identifier"].tolist()

perf_files = [
    str(p) for p in ANALYTICAL_DIR.glob("loan_month_*.parquet")
    if not p.name.endswith("_test.parquet")
]

print("Loans to investigate:", len(loan_ids))
print("Performance files:", len(perf_files))


# --------------------------------------------------
# 2. RETRIEVE MONTHLY PERFORMANCE HISTORY
# --------------------------------------------------

con = duckdb.connect()

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

try:
    history_no_90 = con.execute("""
        SELECT
            loan_identifier,
            period,
            TRIM(current_loan_delinquency_status)
                AS delinquency_status,
            TRY_CAST(
                TRIM(current_loan_delinquency_status)
                AS INTEGER
            ) AS numeric_delinquency,
            TRY_CAST(current_actual_upb AS DOUBLE)
                AS current_upb
        FROM read_parquet(?)
        WHERE loan_identifier IN (SELECT UNNEST(?))
        ORDER BY loan_identifier, period
    """, [perf_files, loan_ids]).df()

finally:
    con.close()

# Attach event codes
history_no_90 = history_no_90.merge(
    no_90plus[["loan_identifier", "event_code"]],
    on="loan_identifier",
    how="left",
    validate="many_to_one"
)

print("Retrieved monthly records:", len(history_no_90))
display(history_no_90.head(10))


# --------------------------------------------------
# 3. DELINQUENCY STATUS DISTRIBUTION
# --------------------------------------------------

status_distribution = (
    history_no_90
    .groupby(
        ["event_code", "delinquency_status"],
        dropna=False
    )
    .agg(
        performance_records=("loan_identifier", "size"),
        unique_loans=("loan_identifier", "nunique")
    )
    .reset_index()
    .sort_values(
        ["event_code", "delinquency_status"]
    )
)

display(status_distribution)


# --------------------------------------------------
# 4. LOAN-LEVEL SEVERITY SUMMARY
# --------------------------------------------------

loan_severity = (
    history_no_90
    .groupby(["loan_identifier", "event_code"])
    .agg(
        max_numeric_delinquency=(
            "numeric_delinquency", "max"
        ),
        first_observed_status=(
            "delinquency_status", "first"
        ),
        last_observed_status=(
            "delinquency_status", "last"
        ),
        observation_months=("period", "nunique"),
        has_60_89_DPD=(
            "numeric_delinquency",
            lambda x: x.eq(2).any()
        ),
        has_30_59_DPD=(
            "numeric_delinquency",
            lambda x: x.eq(1).any()
        ),
        has_REO=(
            "delinquency_status",
            lambda x: x.eq("RA").any()
        ),
        has_unknown_status=(
            "delinquency_status",
            lambda x: x.eq("XX").any()
        )
    )
    .reset_index()
)

display(loan_severity.head(20))


# --------------------------------------------------
# 5. SUMMARY BY EVENT CODE
# --------------------------------------------------

severity_summary = (
    loan_severity
    .groupby("event_code")
    .agg(
        total_loans=("loan_identifier", "nunique"),
        loans_with_60_89_DPD=("has_60_89_DPD", "sum"),
        loans_with_30_59_DPD=("has_30_59_DPD", "sum"),
        loans_with_REO=("has_REO", "sum"),
        loans_with_unknown_status=(
            "has_unknown_status", "sum"
        )
    )
    .reset_index()
)

display(severity_summary)


# --------------------------------------------------
# 6. FINAL OBSERVED STATUS DISTRIBUTION
# --------------------------------------------------

final_status_distribution = (
    loan_severity
    .groupby(
        ["event_code", "last_observed_status"],
        dropna=False
    )
    .size()
    .reset_index(name="loan_count")
    .sort_values(
        ["event_code", "loan_count"],
        ascending=[True, False]
    )
)

display(final_status_distribution)


Loans to investigate: 3195
Performance files: 28


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Retrieved monthly records: 145246


,loan_identifier,period,delinquency_status,numeric_delinquency,current_upb,event_code
0,F06Q10000904,200602,00,0,80000.00,03
1,F06Q10000904,200603,00,0,80000.00,03
2,F06Q10000904,200604,00,0,79000.00,03
3,F06Q10000904,200605,00,0,79000.00,03
4,F06Q10000904,200606,00,0,79000.00,03
5,F06Q10000904,200607,00,0,79000.00,03
6,F06Q10000904,200608,00,0,79000.00,03
7,F06Q10000904,200609,00,0,79096.69,03
8,F06Q10000904,200610,00,0,79021.88,03
9,F06Q10000904,200611,00,0,78946.67,03


,event_code,delinquency_status,performance_records,unique_loans
0,03,00,122955,2916
1,03,01,2893,1591
2,03,02,1304,1169
3,09,00,15163,279
4,09,01,466,237
5,09,02,209,164
6,09,RA,2256,278


,loan_identifier,event_code,max_numeric_delinquency,first_observed_status,last_observed_status,observation_months,has_60_89_DPD,has_30_59_DPD,has_REO,has_unknown_status
0,F06Q10000904,03,0,00,00,77,False,False,False,False
1,F06Q10001324,03,1,00,00,51,False,True,False,False
2,F06Q10003595,03,0,00,00,70,False,False,False,False
3,F06Q10004931,03,2,00,02,58,True,True,False,False
4,F06Q10009407,03,0,00,00,64,False,False,False,False
5,F06Q10011550,03,2,00,02,52,True,True,False,False
6,F06Q10012398,03,0,00,00,59,False,False,False,False
7,F06Q10013792,03,0,00,00,33,False,False,False,False
8,F06Q10014353,03,2,00,02,52,True,True,False,False
9,F06Q10015787,03,0,00,00,48,False,False,False,False


,event_code,total_loans,loans_with_60_89_DPD,loans_with_30_59_DPD,loans_with_REO,loans_with_unknown_status
0,03,2916,1169,1591,0,0
1,09,279,164,237,278,0


,event_code,last_observed_status,loan_count
0,03,00,1575
2,03,02,1103
1,03,01,238
4,09,RA,278
3,09,00,1


In [33]:

# Cell 29 — Delinquency-to-Termination Timeline

import pandas as pd

# history_no_90 and no_90plus are retained from Cell 28.

# 1. Find the most severe numeric delinquency status
loan_timeline = (
    history_no_90
    .groupby(["loan_identifier", "event_code"])
    .agg(
        first_period=("period", "min"),
        last_period=("period", "max"),
        max_delinquency=(
            "numeric_delinquency", "max"
        ),
        first_30plus_period=(
            "period",
            lambda x: x.iloc[0]
        ),
        last_status=("delinquency_status", "last"),
        has_REO=("delinquency_status",
                 lambda x: x.eq("RA").any())
    )
    .reset_index()
)

# 2. Find first month at each delinquency threshold
thresholds = (
    history_no_90
    .loc[history_no_90["numeric_delinquency"].notna()]
    .assign(
        numeric_delinquency=lambda x:
            x["numeric_delinquency"].astype(int)
    )
    .groupby(["loan_identifier", "event_code"])
    .agg(
        first_30_59=(
            "period",
            lambda x: x[
                history_no_90.loc[x.index,
                    "numeric_delinquency"].eq(1)
            ].min()
        ),
        first_60_89=(
            "period",
            lambda x: x[
                history_no_90.loc[x.index,
                    "numeric_delinquency"].eq(2)
            ].min()
        )
    )
    .reset_index()
)

loan_timeline = loan_timeline.merge(
    thresholds,
    on=["loan_identifier", "event_code"],
    how="left",
    validate="one_to_one"
)

# 3. Attach event dates
loan_timeline = loan_timeline.merge(
    no_90plus[
        [
            "loan_identifier",
            "event_reporting_period",
            "zero_balance_effective_date"
        ]
    ],
    on="loan_identifier",
    how="left",
    validate="one_to_one"
)

# 4. Display results
display(
    loan_timeline.groupby("event_code").agg(
        total_loans=("loan_identifier", "nunique"),
        loans_reaching_30_59=("first_30_59",
                              lambda x: x.notna().sum()),
        loans_reaching_60_89=("first_60_89",
                              lambda x: x.notna().sum()),
        loans_with_REO=("has_REO", "sum"),
        loans_ending_current=("last_status",
                              lambda x: x.eq("00").sum()),
        loans_ending_30_59=("last_status",
                            lambda x: x.eq("01").sum()),
        loans_ending_60_89=("last_status",
                           lambda x: x.eq("02").sum()),
        loans_ending_REO=("last_status",
                          lambda x: x.eq("RA").sum())
    ).reset_index()
)

# 5. Inspect individual timelines
display(loan_timeline.head(30))


,event_code,total_loans,loans_reaching_30_59,loans_reaching_60_89,loans_with_REO,loans_ending_current,loans_ending_30_59,loans_ending_60_89,loans_ending_REO
0,03,2916,1591,1169,0,1575,238,1103,0
1,09,279,237,164,278,1,0,0,278


,loan_identifier,event_code,first_period,last_period,max_delinquency,first_30plus_period,last_status,has_REO,first_30_59,first_60_89,event_reporting_period,zero_balance_effective_date
0,F06Q10000904,03,200602,201206,0,200602,00,False,NaN,NaN,201206,201206
1,F06Q10001324,03,200602,201004,1,200602,00,False,200702,NaN,201004,201004
2,F06Q10003595,03,200602,201111,0,200602,00,False,NaN,NaN,201111,201111
3,F06Q10004931,03,200602,201011,2,200602,02,False,201010,201011,201011,201011
4,F06Q10009407,03,200602,201105,0,200602,00,False,NaN,NaN,201105,201105
5,F06Q10011550,03,200602,201005,2,200602,02,False,200908,201005,201005,201005
6,F06Q10012398,03,200602,201012,0,200602,00,False,NaN,NaN,201012,201012
7,F06Q10013792,03,200602,200810,0,200602,00,False,NaN,NaN,200810,200810
8,F06Q10014353,03,200602,201005,2,200602,02,False,201004,201005,201005,201005
9,F06Q10015787,03,200602,201001,0,200602,00,False,NaN,NaN,201001,201001
